# TinySolverDx demo
Batched small-matrix Cholesky / LU on NumPy arrays. Build first: `make setup build` and run with `PYTHONPATH=python`.

In [ ]:
import time, numpy as np, scipy.linalg as sla
import tinysolverdx as tsdx
from tinysolverdx import matrices
print('threads', tsdx.num_threads(), 'openmp', tsdx.has_openmp(), 'cuda', tsdx.has_cuda())

## Solve 100k SPD systems of size 16

In [ ]:
rng = np.random.default_rng(0)
A = matrices.spd(rng, 100_000, 16, kappa=1e3).astype(np.float32)
b = rng.standard_normal((100_000, 16)).astype(np.float32)
t = time.perf_counter(); x, info = tsdx.posv(A, b); t_tsdx = time.perf_counter() - t
t = time.perf_counter(); xl = np.stack([np.linalg.solve(a, v) for a, v in zip(A[:5000], b[:5000])]); t_loop = (time.perf_counter() - t) * 20
print(f'tsdx {t_tsdx*1e3:.0f} ms vs looped numpy ~{t_loop*1e3:.0f} ms (extrapolated from 5000)')
print('all ok:', (info == 0).all(), ' max rel. diff to numpy =', (abs(x[:5000] - xl).max() / abs(xl).max()))

## Accuracy vs. a long-double reference, across precisions

In [ ]:
A64 = matrices.general(rng, 200, 16, kappa=1e2)
b64 = rng.standard_normal((200, 16))
for prec in ['fp16', 'bf16', 'fp32', 'fp64']:
    Ap = tsdx.round_to(A64, prec)
    LU, piv, info = tsdx.lu(Ap, precision=prec)
    bw = tsdx.lu_backward_error(Ap, LU.astype(np.float64), piv)
    x, _ = tsdx.gesv(Ap, b64, precision=prec)
    fw = tsdx.solve_forward_error(Ap, x.astype(np.float64), b64)
    print(f'{prec}: u={tsdx.unit_roundoff(prec):.1e}  backward={np.median(bw):.1e}  forward={np.median(fw):.1e}')

The full sweeps (backward/forward error vs. condition number, per precision) are produced by `make report` and live in `docs/report/REPORT.md`.